In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Initialize SparkSession
spark=SparkSession.builder \
    .appName("BlogFeedback Data Cleaning") \
    .getOrCreate()

## Step 1. Read Raw Data and Rename Columns
Read the raw CSV data from hdfs (281 columns, no header) and count rows and columns. Set the last column as target and name the rest feature_0 to feature_279.

In [9]:
# 2. Read raw data (281 columns, tab-separated, no header, all numeric)
input_path="/user/maria_dev/blog_project/raw/blogData_train.csv"
raw_df=spark.read \
    .option("sep",",") \
    .option("header","false") \
    .option("inferSchema","true") \
    .csv(input_path)
row_count=raw_df.count()
col_count=len(raw_df.columns)
print("Raw data row count:",row_count,", column count:",col_count)

# 3. Rename columns: last column is target, rest are features
num_columns=len(raw_df.columns)
feature_names=["feature_{}".format(i) for i in range(num_columns - 1)]
target_name="target"
df=raw_df.toDF(*(feature_names + [target_name]))
# 4. Separate feature columns and target column (for easier subsequent analysis)
feature_cols=[c for c in df.columns if c != "target"]
target_col="target"

('Raw data row count:', 52397, ', column count:', 281)


## Step 3. Remove Constant Feature Columns
Detect columns with standard deviation of 0 (constant values) and drop them from the dataset to reduce useless features.

In [10]:
from pyspark.sql.functions import col,count,when,stddev

# 5. Detect and remove constant columns (columns with standard deviation of 0)
print("Detecting constant columns..")
agg_exprs=[stddev(col(c)).alias(c) for c in feature_cols]
std_devs=df.select(agg_exprs).collect()[0].asDict()
const_cols=[c for c,dev in std_devs.items() if dev == 0.0]
if const_cols:
    print("constant columns detected(stddev=0):",const_cols)
    df=df.drop(*const_cols)
    # Update  list
    feature_cols=[c for c in df.columns if c != "target"]
else:
    print("No constant columns")
print("After remove constant column,remaining column:",len(df.columns),"\n")


Detecting constant columns..
('constant columns detected(stddev=0):', ['feature_32', 'feature_37', 'feature_277', 'feature_12'])
('After remove constant column,remaining column:', 277, '\n')


## Step 4. Check for Missing Values
Check for null values in the dataset. The result shows no missing values in any column.

In [4]:
# 6. Check for missing values (only show columns with missing values)
print("Checking for null values")
null_counts=df.select([count(when(col(c).isNull(),c)).alias(c) for c in df.columns])
null_row=null_counts.collect()[0]
missing_cols={c: null_row[c] for c in df.columns if null_row[c] > 0}
if not missing_cols:
    print("No missing value\n")


Checking for missing values...
No missing values in any columns



## Step 5. Remove Duplicate Rows and Cast Target Column
Remove exact duplicate rows (3194 rows removed). Cast the target column to integer type for easier regression analysis.the we save the csv file to hdfs.

In [4]:
# 7. Remove exact duplicate rows
before_dedup=df.count()
df=df.dropDuplicates()
after_dedup=df.count()
print("Rows before remove duplicate:",before_dedup,", rows after:",after_dedup,", duplicate rows removed:",before_dedup - after_dedup)
# 8. Cast target column to integer (for easier regression analysis later)
df=df.withColumn("target",df["target"].cast("int"))

('Rows before remove duplicate:', 52397, ', rows after:', 49203, ', duplicate rows removed:', 3194)


In [5]:
output_path="/user/maria_dev/blog_project/blogData_train_clean.csv"
df.write \
    .mode("overwrite") \
    .option("header","true") \
    .option("sep",",") \
    .csv(output_path)
print("Cleaned CSV saved to",output_path)

('Cleaned CSV saved to', '/user/maria_dev/blog_project/blogData_train_clean.csv')


## Step 6. Get Test File List
Scan all CSV files in the raw data directory (61 files total).Exclude the training file,leaving 60 test files for the final prediction evaluation.

In [6]:
# Process all CSV files except training file
train_columns = df.columns
data_path = "/user/maria_dev/blog_project/raw/"

# Get all CSV files in the directory
all_files_rdd = spark.sparkContext.wholeTextFiles(data_path + "*.csv")
all_files = [path.split("/")[-1] for path,content in all_files_rdd.collect()]

# Exclude training file
train_file = "blogData_train.csv"
test_files = [f for f in all_files if f != train_file]

print("found file number:", len(all_files))
print("Test files :", len(test_files))
print("First few test files:", test_files[:3])


('found file number:', 61)
('Test files :', 60)
('First few test files:', [u'blogData_test-2012.02.01.00_00.csv', u'blogData_test-2012.02.02.00_00.csv', u'blogData_test-2012.02.03.00_00.csv'])


## Step 7. Clean and Save Test Files
Read each 60 test files one by one, rename columns to match the training data, keep only the 277 columns present in the cleaned training data

In [7]:
for test_file in test_files:
    print("\nProcess:", test_file)
    
    # Read test file
    test_df = spark.read.option("sep",",").option("header","false").option("inferSchema","true").csv(data_path + test_file)
    print("Raw test columns:", len(test_df.columns))
    
    # Rename columns to match training data
    num_cols = len(test_df.columns)
    test_df = test_df.toDF(*(["feature_{}".format(i) for i in range(num_cols - 1)] + ["target"]))
    
    # Keep only columns that exist in cleaned training data
    test_df = test_df.select(*[c for c in train_columns if c in test_df.columns])
    print("After filtering, test columns:", len(test_df.columns))
    

    # Save
    output_path = "/user/maria_dev/blog_project/clean_test/" + test_file.replace(".csv", "_clean.csv")
    test_df.write.mode("overwrite").option("header","true").option("sep",",").csv(output_path)
    print("Save:", output_path)

('\nProcess:', u'blogData_test-2012.02.01.00_00.csv')
('Raw test columns:', 281)
('After filtering, test columns:', 277)
('Save:', u'/user/maria_dev/blog_project/clean_test/blogData_test-2012.02.01.00_00_clean.csv')
('\nProcess:', u'blogData_test-2012.02.02.00_00.csv')
('Raw test columns:', 281)
('After filtering, test columns:', 277)
('Save:', u'/user/maria_dev/blog_project/clean_test/blogData_test-2012.02.02.00_00_clean.csv')
('\nProcess:', u'blogData_test-2012.02.03.00_00.csv')
('Raw test columns:', 281)
('After filtering, test columns:', 277)
('Save:', u'/user/maria_dev/blog_project/clean_test/blogData_test-2012.02.03.00_00_clean.csv')
('\nProcess:', u'blogData_test-2012.02.04.00_00.csv')
('Raw test columns:', 281)
('After filtering, test columns:', 277)
('Save:', u'/user/maria_dev/blog_project/clean_test/blogData_test-2012.02.04.00_00_clean.csv')
('\nProcess:', u'blogData_test-2012.02.05.00_00.csv')
('Raw test columns:', 281)
('After filtering, test columns:', 277)
('Save:', u'/us

('After filtering, test columns:', 277)
('Save:', u'/user/maria_dev/blog_project/clean_test/blogData_test-2012.03.10.00_00_clean.csv')
('\nProcess:', u'blogData_test-2012.03.11.00_00.csv')
('Raw test columns:', 281)
('After filtering, test columns:', 277)
('Save:', u'/user/maria_dev/blog_project/clean_test/blogData_test-2012.03.11.00_00_clean.csv')
('\nProcess:', u'blogData_test-2012.03.12.00_00.csv')
('Raw test columns:', 281)
('After filtering, test columns:', 277)
('Save:', u'/user/maria_dev/blog_project/clean_test/blogData_test-2012.03.12.00_00_clean.csv')
('\nProcess:', u'blogData_test-2012.03.13.00_00.csv')
('Raw test columns:', 281)
('After filtering, test columns:', 277)
('Save:', u'/user/maria_dev/blog_project/clean_test/blogData_test-2012.03.13.00_00_clean.csv')
('\nProcess:', u'blogData_test-2012.03.14.00_00.csv')
('Raw test columns:', 281)
('After filtering, test columns:', 277)
('Save:', u'/user/maria_dev/blog_project/clean_test/blogData_test-2012.03.14.00_00_clean.csv')
(